# 02 Preprocessing Pipeline

Purpose: clean the raw data, use all non-ID predictor features, scale features, and save the 70:30 train-test split.

## 1. Import Libraries and Paths

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA = PROJECT_ROOT / "data" / "raw" / "PCOS_Data.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Dataset

In [ ]:
df = pd.read_csv(RAW_DATA)
df.columns = df.columns.str.strip()
print(df.shape)
display(df.head())

(541, 44)


,Sl. No,Patient File No.,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),...,Pimples(Y/N),Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm)
0,1,1,0,28,44.6,152.0,19.3,15,78,22,...,0,1.0,0,110,80,3,3,18.0,18.0,8.5
1,2,2,0,36,65.0,161.5,24.9,15,74,20,...,0,0.0,0,120,70,3,5,15.0,14.0,3.7
2,3,3,1,33,68.8,165.0,25.3,11,72,18,...,1,1.0,0,120,80,13,15,18.0,20.0,10.0
3,4,4,0,37,65.0,148.0,29.7,13,72,20,...,0,0.0,0,120,70,2,2,15.0,14.0,7.5
4,5,5,0,25,52.0,161.0,20.1,11,72,18,...,0,0.0,0,120,80,3,4,16.0,14.0,7.0


## 3. Remove Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows")

Removed 0 duplicate rows


## 4. Drop Unwanted ID Columns

In [ ]:
target_col = "PCOS (Y/N)"
id_cols = ["Sl. No", "Patient File No."]
df = df.drop(columns=[c for c in id_cols if c in df.columns])
print(f"Columns after dropping IDs: {df.shape[1]}")

Columns after dropping IDs: 42


## 5. Convert Numeric Columns

In [ ]:
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

conversion_missing = df.isna().sum().sort_values(ascending=False)
display(conversion_missing[conversion_missing > 0].to_frame("Missing after numeric conversion"))

,Missing after numeric conversion
II beta-HCG(mIU/mL),1
Marraige Status (Yrs),1
Fast food (Y/N),1
AMH(ng/mL),1


## 6. Separate Target and Features

In [ ]:
y = df[target_col].astype(int)
X = df.drop(columns=[target_col])
print(f"Total predictors used: {X.shape[1]}")
print(f"Target distribution:\n{y.value_counts(normalize=True).rename('proportion')}")
display(X.head())

Total predictors used: 41
Target distribution:
PCOS (Y/N)
0    0.672828
1    0.327172
Name: proportion, dtype: float64


,Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),Hb(g/dl),Cycle(R/I),Cycle length(days),...,Pimples(Y/N),Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm)
0,28,44.6,152.0,19.3,15,78,22,10.48,2,5,...,0,1.0,0,110,80,3,3,18.0,18.0,8.5
1,36,65.0,161.5,24.9,15,74,20,11.70,2,5,...,0,0.0,0,120,70,3,5,15.0,14.0,3.7
2,33,68.8,165.0,25.3,11,72,18,11.80,2,5,...,1,1.0,0,120,80,13,15,18.0,20.0,10.0
3,37,65.0,148.0,29.7,13,72,20,12.00,2,5,...,0,0.0,0,120,70,2,2,15.0,14.0,7.5
4,25,52.0,161.0,20.1,11,72,18,10.00,2,5,...,0,0.0,0,120,80,3,4,16.0,14.0,7.0


## 7. Train-Test Split: 70:30 Stratified

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print(X_train_raw.shape, X_test_raw.shape, y_train.shape, y_test.shape)

(378, 41) (163, 41) (378,) (163,)


## 8. Missing Value Handling and Scaling

In [ ]:
numeric_features = X.columns.tolist()
preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

X_train_processed = pd.DataFrame(
    preprocess.fit_transform(X_train_raw),
    columns=numeric_features,
    index=X_train_raw.index,
)
X_test_processed = pd.DataFrame(
    preprocess.transform(X_test_raw),
    columns=numeric_features,
    index=X_test_raw.index,
)

print("Missing values after preprocessing:", int(X_train_processed.isna().sum().sum() + X_test_processed.isna().sum().sum()))
display(X_train_processed.describe().T.head())

Missing values after preprocessing: 0


,count,mean,std,min,25%,50%,75%,max
Age (yrs),378.0,-2.443665e-16,1.001325,-2.126490,-0.827323,-0.084942,0.657439,2.884583
Weight (Kg),378.0,5.733215e-16,1.001325,-2.503143,-0.650657,-0.033162,0.496120,4.289305
Height(Cm),378.0,-1.409807e-16,1.001325,-2.798493,-0.762502,-0.083838,0.594825,3.988143
BMI,378.0,-1.315820e-16,1.001325,-2.871622,-0.653884,0.016312,0.589024,3.586626
Blood Group,378.0,2.678633e-16,1.001325,-1.524764,-0.434204,0.111075,0.656355,2.292194


## 9. Save Processed Data and Pipeline

In [ ]:
X_train_raw.to_csv(PROCESSED_DIR / "X_train_raw_split.csv", index=False)
X_test_raw.to_csv(PROCESSED_DIR / "X_test_raw_split.csv", index=False)
X_train_processed.to_csv(PROCESSED_DIR / "X_train_processed.csv", index=False)
X_test_processed.to_csv(PROCESSED_DIR / "X_test_processed.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False, header=["PCOS"])
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False, header=["PCOS"])
joblib.dump(preprocess, MODELS_DIR / "preprocessing_pipeline.joblib")

metadata = {
    "target": target_col,
    "dropped_id_columns": id_cols,
    "predictor_count": int(X.shape[1]),
    "split": "70:30 stratified",
    "random_state": 42,
    "features": numeric_features,
}
(PROCESSED_DIR / "preprocessing_metadata.json").write_text(json.dumps(metadata, indent=2))
print("Saved processed train/test files and preprocessing pipeline.")

Saved processed train/test files and preprocessing pipeline.
